# 06 - TF-IDF Baseline (Corrected Labels)

Train a TF-IDF + classifier pipeline on the Sonnet-corrected data and compare
against all other models evaluated on the same val set.

**Strategy:**
- Train on `data/corrected/train.parquet` (78,357 rows)
- Evaluate on `data/corrected/val.parquet` (9,810 rows) -- same val set as MLP and ModernBERT
- Text input: the `text` column (format: `domain | title | keywords`)
- Compare 3 classifiers: Logistic Regression, SGD, LinearSVC
- Final comparison table against v2 MLP (84.9%) and v1 ModernBERT (61.3%)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import time
import json
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, top_k_accuracy_score,
    f1_score
)

PROJECT_DIR = Path('.').resolve().parent.parent
DATA_DIR = PROJECT_DIR / 'data' / 'corrected'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
MODEL_DIR = PROJECT_DIR / 'models'

print(f'Project: {PROJECT_DIR.name}')

Project: IAB_LLM_Distillation_Classification


In [2]:
# Load corrected data (same splits used by all other models)
train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
val_df = pd.read_parquet(DATA_DIR / 'val.parquet')
test_df = pd.read_parquet(DATA_DIR / 'test.parquet')

print(f'Train: {len(train_df):,} rows')
print(f'Val:   {len(val_df):,} rows')
print(f'Test:  {len(test_df):,} rows')
print(f'\nAll models (MLP, ModernBERT, TF-IDF) use the same val set for comparison.')

Train: 78,357 rows
Val:   9,810 rows
Test:  9,794 rows

All models (MLP, ModernBERT, TF-IDF) use the same val set for comparison.


In [3]:
# Encode labels
le = LabelEncoder()
le.fit(sorted(train_df['tier1'].unique()))

train_labels = le.transform(train_df['tier1'])
val_labels = le.transform(val_df['tier1'])
test_labels = le.transform(test_df['tier1'])

CATEGORIES = le.classes_.tolist()
NUM_CLASSES = len(CATEGORIES)
print(f'Classes: {NUM_CLASSES}')
print(f'Label distribution (train, top 5):')
for cat, count in train_df['tier1'].value_counts().head(5).items():
    print(f'  {cat:<30} {count:>6,} ({count/len(train_df)*100:.1f}%)')

Classes: 27
Label distribution (train, top 5):
  Shopping                       13,175 (16.8%)
  Arts & Entertainment            8,582 (11.0%)
  Business & Industrial           7,231 (9.2%)
  Computers & Electronics         4,679 (6.0%)
  Jobs & Education                3,555 (4.5%)


In [4]:
# Prepare text -- use the enriched 'text' column (domain | title | keywords)
train_texts = train_df['text'].fillna(train_df['domain_clean']).tolist()
val_texts = val_df['text'].fillna(val_df['domain_clean']).tolist()
test_texts = test_df['text'].fillna(test_df['domain_clean']).tolist()

print(f'Sample texts:')
for t in train_texts[:5]:
    print(f'  {t[:100]}')

Sample texts:
  allesiszoleuk.nl | welkom | allesiszoleuk | online store, retail shop, Dutch webshop, customer messa
  tricom.se | tricom | tricom, telecommunications, communications, technology services, business solut
  farmaciarodriguezprada.es | farmacia rodriguez prada abiertos todo el día. envío express. parafarmac
  g-music.com.tw | 玫瑰大眾購物網 | music store, online shopping, records, CDs, entertainment, music retail, 
  bloggotof.com | bloggotof | culture, diversity, geek, music, media, LGBTQI, entertainment, blog, lif


## TF-IDF Vectorization

We use unigrams + bigrams with sublinear TF scaling. The Sonnet-generated keywords
(e.g., "online store, electronics, gadgets") are highly discriminative bigrams that
TF-IDF should exploit well.

In [5]:
# TF-IDF vectorization
start = time.time()
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.8,
    strip_accents='unicode',
    dtype=np.float32,
)

X_train = tfidf.fit_transform(train_texts)
X_val = tfidf.transform(val_texts)
X_test = tfidf.transform(test_texts)
elapsed = time.time() - start

print(f'TF-IDF vectorization: {elapsed:.1f}s')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,}')
print(f'Train matrix: {X_train.shape} (sparse, {X_train.nnz:,} non-zeros)')
print(f'Val matrix: {X_val.shape}')
print(f'Sparsity: {1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]):.4%}')

# Show top features
feature_names = tfidf.get_feature_names_out()
print(f'\nSample features (first 20): {feature_names[:20].tolist()}')
print(f'Sample bigrams: {[f for f in feature_names if " " in f][:15]}')

TF-IDF vectorization: 3.4s
Vocabulary size: 30,000
Train matrix: (78357, 30000) (sparse, 2,055,513 non-zeros)
Val matrix: (9810, 30000)
Sparsity: 99.9126%

Sample features (first 20): ['000', '01', '03', '05', '05 2022', '10', '10 000', '10 software', '100', '1000', '1001', '101', '1080p', '11', '12', '12 education', '123', '13', '14', '15']
Sample bigrams: ['05 2022', '10 000', '10 software', '12 education', '19th century', '1a a1', '2021 2022', '2022 2023', '2022 fifa', '2022 football', '2022 online', '2022 top', '2022 world', '20th century', '21st century']


## Model Comparison: 3 Classifiers

We compare Logistic Regression, SGD (log loss), and LinearSVC -- all suitable for
high-dimensional sparse features. Each is trained with class weighting to handle
the 253x class imbalance.

In [6]:
# Train and evaluate 3 classifiers
results = []

# 1. Logistic Regression
print('Training Logistic Regression...')
start = time.time()
lr_model = LogisticRegression(
    max_iter=1000, C=1.0, solver='lbfgs',
    class_weight='balanced', n_jobs=-1
)
lr_model.fit(X_train, train_labels)
lr_time = time.time() - start

lr_val_preds = lr_model.predict(X_val)
lr_val_probs = lr_model.predict_proba(X_val)
lr_top1 = accuracy_score(val_labels, lr_val_preds)
lr_top3 = top_k_accuracy_score(val_labels, lr_val_probs, k=3, labels=np.arange(NUM_CLASSES))
lr_f1 = f1_score(val_labels, lr_val_preds, average='macro')
print(f'  Top-1: {lr_top1:.1%}, Top-3: {lr_top3:.1%}, Macro-F1: {lr_f1:.3f}, Time: {lr_time:.1f}s')
results.append(('Logistic Regression', lr_top1, lr_top3, lr_f1, lr_time))

# 2. SGD Classifier (log loss)
print('\nTraining SGD Classifier...')
start = time.time()
sgd_base = SGDClassifier(
    loss='log_loss', alpha=1e-4, max_iter=1000,
    class_weight='balanced', random_state=42, n_jobs=-1
)
sgd_model = CalibratedClassifierCV(sgd_base, cv=3, method='sigmoid')
sgd_model.fit(X_train, train_labels)
sgd_time = time.time() - start

sgd_val_preds = sgd_model.predict(X_val)
sgd_val_probs = sgd_model.predict_proba(X_val)
sgd_top1 = accuracy_score(val_labels, sgd_val_preds)
sgd_top3 = top_k_accuracy_score(val_labels, sgd_val_probs, k=3, labels=np.arange(NUM_CLASSES))
sgd_f1 = f1_score(val_labels, sgd_val_preds, average='macro')
print(f'  Top-1: {sgd_top1:.1%}, Top-3: {sgd_top3:.1%}, Macro-F1: {sgd_f1:.3f}, Time: {sgd_time:.1f}s')
results.append(('SGD (Calibrated)', sgd_top1, sgd_top3, sgd_f1, sgd_time))

# 3. LinearSVC + Calibration
print('\nTraining LinearSVC...')
start = time.time()
svc_base = LinearSVC(
    C=1.0, max_iter=2000, class_weight='balanced', random_state=42
)
svc_model = CalibratedClassifierCV(svc_base, cv=3, method='sigmoid')
svc_model.fit(X_train, train_labels)
svc_time = time.time() - start

svc_val_preds = svc_model.predict(X_val)
svc_val_probs = svc_model.predict_proba(X_val)
svc_top1 = accuracy_score(val_labels, svc_val_preds)
svc_top3 = top_k_accuracy_score(val_labels, svc_val_probs, k=3, labels=np.arange(NUM_CLASSES))
svc_f1 = f1_score(val_labels, svc_val_preds, average='macro')
print(f'  Top-1: {svc_top1:.1%}, Top-3: {svc_top3:.1%}, Macro-F1: {svc_f1:.3f}, Time: {svc_time:.1f}s')
results.append(('LinearSVC (Calibrated)', svc_top1, svc_top3, svc_f1, svc_time))

Training Logistic Regression...


  Top-1: 90.4%, Top-3: 99.4%, Macro-F1: 0.878, Time: 6.2s

Training SGD Classifier...


  Top-1: 89.7%, Top-3: 99.2%, Macro-F1: 0.861, Time: 1.7s

Training LinearSVC...


  Top-1: 91.6%, Top-3: 99.3%, Macro-F1: 0.880, Time: 12.5s


In [7]:
# Summary of TF-IDF classifiers
print(f'TF-IDF CLASSIFIER COMPARISON (val set)')
print(f'=' * 70)
print(f'{"Classifier":<25} | {"Top-1":>7} | {"Top-3":>7} | {"Macro-F1":>8} | {"Train Time":>10}')
print('-' * 70)
for name, top1, top3, f1, t in results:
    print(f'{name:<25} | {top1:>6.1%} | {top3:>6.1%} | {f1:>8.3f} | {t:>8.1f}s')

# Select best
best_idx = np.argmax([r[1] for r in results])
best_name, best_top1, best_top3, best_f1, _ = results[best_idx]
print(f'\nBest TF-IDF model: {best_name} ({best_top1:.1%} top-1)')

TF-IDF CLASSIFIER COMPARISON (val set)
Classifier                |   Top-1 |   Top-3 | Macro-F1 | Train Time
----------------------------------------------------------------------
Logistic Regression       |  90.4% |  99.4% |    0.878 |      6.2s
SGD (Calibrated)          |  89.7% |  99.2% |    0.861 |      1.7s
LinearSVC (Calibrated)    |  91.6% |  99.3% |    0.880 |     12.5s

Best TF-IDF model: LinearSVC (Calibrated) (91.6% top-1)


In [8]:
# Detailed per-category report for best model
# Use the best model's predictions
if best_idx == 0:
    best_preds, best_probs = lr_val_preds, lr_val_probs
elif best_idx == 1:
    best_preds, best_probs = sgd_val_preds, sgd_val_probs
else:
    best_preds, best_probs = svc_val_preds, svc_val_probs

print(f'PER-CATEGORY REPORT ({best_name})')
print(f'=' * 80)
print(classification_report(val_labels, best_preds, target_names=CATEGORIES, digits=3))

PER-CATEGORY REPORT (LinearSVC (Calibrated))
                         precision    recall  f1-score   support

                  Adult      0.984     0.944     0.963       195
   Arts & Entertainment      0.930     0.932     0.931      1088
       Autos & Vehicles      0.946     0.960     0.953       401
       Beauty & Fitness      0.934     0.908     0.921       251
     Books & Literature      0.898     0.865     0.881       274
  Business & Industrial      0.888     0.919     0.904       866
Computers & Electronics      0.923     0.918     0.920       584
                Finance      0.983     0.919     0.950       124
           Food & Drink      0.940     0.940     0.940       251
                  Games      0.925     0.949     0.937       234
                 Health      0.944     0.955     0.950       445
      Hobbies & Leisure      0.861     0.769     0.812       290
          Home & Garden      0.878     0.928     0.902       374
     Internet & Telecom      0.898     0.876

In [9]:
# Also evaluate on test set for final numbers
if best_idx == 0:
    test_preds = lr_model.predict(X_test)
    test_probs = lr_model.predict_proba(X_test)
elif best_idx == 1:
    test_preds = sgd_model.predict(X_test)
    test_probs = sgd_model.predict_proba(X_test)
else:
    test_preds = svc_model.predict(X_test)
    test_probs = svc_model.predict_proba(X_test)

test_top1 = accuracy_score(test_labels, test_preds)
test_top3 = top_k_accuracy_score(test_labels, test_probs, k=3, labels=np.arange(NUM_CLASSES))
test_f1 = f1_score(test_labels, test_preds, average='macro')

print(f'TEST SET RESULTS ({best_name})')
print(f'=' * 50)
print(f'  Top-1 Accuracy: {test_top1:.1%}')
print(f'  Top-3 Accuracy: {test_top3:.1%}')
print(f'  Macro F1: {test_f1:.3f}')
print(f'\nVal vs Test consistency: {abs(best_top1 - test_top1)*100:.1f}pp difference (should be <2pp)')

TEST SET RESULTS (LinearSVC (Calibrated))
  Top-1 Accuracy: 91.7%
  Top-3 Accuracy: 99.2%
  Macro F1: 0.902

Val vs Test consistency: 0.1pp difference (should be <2pp)


In [10]:
# TF-IDF feature importance: top features per category
print(f'TOP TF-IDF FEATURES PER CATEGORY (from {best_name})')
print(f'=' * 80)

# Get coefficients from the best model
if best_idx == 0:
    coef_model = lr_model
elif best_idx == 1:
    # SGD calibrated -- get base estimators
    coef_model = sgd_model.calibrated_classifiers_[0].estimator
else:
    coef_model = svc_model.calibrated_classifiers_[0].estimator

if hasattr(coef_model, 'coef_'):
    coefs = coef_model.coef_
    for i in range(min(10, NUM_CLASSES)):
        cat = CATEGORIES[i]
        top_indices = np.argsort(coefs[i])[::-1][:8]
        top_features = [feature_names[j] for j in top_indices]
        print(f'  {cat:<28}: {" | ".join(top_features)}')
    print(f'  ... (showing 10 of {NUM_CLASSES} categories)')
else:
    print('  (coefficients not available for this model type)')

TOP TF-IDF FEATURES PER CATEGORY (from LinearSVC (Calibrated))
  Adult                       : adult | adult content | judi | adult entertainment | казино | baccarat | lottery betting | content
  Arts & Entertainment        : entertainment | artistic | creative | music | art | arts | museum | storytelling
  Autos & Vehicles            : vehicle | car | automotive | vehicles | motorcycle | parts | boat | aircraft
  Beauty & Fitness            : beauty | skincare | fitness | hair | cosmetics | style | bodybuilding | facial
  Books & Literature          : literature | books | author | book | scholarly | texts | reading | literary
  Business & Industrial       : industrial | business | commercial | industrial equipment | services | marketing | industrial supplies | manufacturing
  Computers & Electronics     : computer | electronics | software | electronic | technology | computing | computers | laptop
  Finance                     : financial | insurance | investment | tax | loans | bankin

In [11]:
# Inference speed comparison
print(f'INFERENCE SPEED')
print(f'=' * 50)

# TF-IDF inference on 1000 samples
sample_texts = val_texts[:1000]
start = time.time()
X_sample = tfidf.transform(sample_texts)
if best_idx == 0:
    _ = lr_model.predict_proba(X_sample)
elif best_idx == 1:
    _ = sgd_model.predict_proba(X_sample)
else:
    _ = svc_model.predict_proba(X_sample)
tfidf_elapsed = time.time() - start
tfidf_per_domain = tfidf_elapsed / 1000 * 1000  # ms

print(f'TF-IDF ({best_name}): {tfidf_per_domain:.2f} ms/domain')
print(f'  (vectorization + classification combined)')
print(f'\nFor comparison:')
print(f'  E5-small + MLP: ~1ms/domain (embedding cached)')
print(f'  ModernBERT: ~8-15ms/domain (full forward pass)')

INFERENCE SPEED
TF-IDF (LinearSVC (Calibrated)): 0.02 ms/domain
  (vectorization + classification combined)

For comparison:
  E5-small + MLP: ~1ms/domain (embedding cached)
  ModernBERT: ~8-15ms/domain (full forward pass)


In [12]:
# Hyperparameter sensitivity: does more features or trigrams help?
print(f'HYPERPARAMETER SENSITIVITY')
print(f'=' * 60)
print(f'{"Config":<35} | {"Top-1":>7} | {"Macro-F1":>8} | {"Time":>6}')
print('-' * 65)

configs = [
    ('20K features, unigrams', {'max_features': 20000, 'ngram_range': (1,1)}),
    ('20K features, uni+bigrams', {'max_features': 20000, 'ngram_range': (1,2)}),
    ('30K features, uni+bigrams', {'max_features': 30000, 'ngram_range': (1,2)}),
    ('50K features, uni+bigrams', {'max_features': 50000, 'ngram_range': (1,2)}),
    ('30K features, uni+bi+trigrams', {'max_features': 30000, 'ngram_range': (1,3)}),
]

for name, params in configs:
    start = time.time()
    tfidf_temp = TfidfVectorizer(
        sublinear_tf=True, min_df=3, max_df=0.8,
        strip_accents='unicode', dtype=np.float32, **params
    )
    X_tr = tfidf_temp.fit_transform(train_texts)
    X_va = tfidf_temp.transform(val_texts)
    lr_temp = LogisticRegression(max_iter=500, C=1.0, solver='lbfgs',
                                  class_weight='balanced', n_jobs=-1)
    lr_temp.fit(X_tr, train_labels)
    preds = lr_temp.predict(X_va)
    acc = accuracy_score(val_labels, preds)
    f1 = f1_score(val_labels, preds, average='macro')
    elapsed = time.time() - start
    print(f'{name:<35} | {acc:>6.1%} | {f1:>8.3f} | {elapsed:>5.1f}s')

HYPERPARAMETER SENSITIVITY
Config                              |   Top-1 | Macro-F1 |   Time
-----------------------------------------------------------------


20K features, unigrams              |  90.2% |    0.876 |   5.5s


20K features, uni+bigrams           |  90.5% |    0.878 |   8.5s


30K features, uni+bigrams           |  90.4% |    0.878 |   9.7s


50K features, uni+bigrams           |  90.5% |    0.880 |  12.9s


30K features, uni+bi+trigrams       |  90.5% |    0.877 |  12.7s


In [13]:
# FULL MODEL COMPARISON TABLE
print(f'\n{"="*80}')
print(f'FULL MODEL COMPARISON (all evaluated on same val set: 9,699 domains)')
print(f'{"="*80}')
print(f'{"Model":<45} | {"Top-1":>7} | {"Top-3":>7} | {"Params":>8} | {"Speed":>8}')
print(f'{"-"*80}')
print(f'{"TF-IDF + " + best_name:<45} | {best_top1*100:.1f}% | {best_top3*100:.1f}% | {"~30K feat":>8} | {"0.1ms":>8}')
print(f'{"v2 MLP (E5 + distillation)":<45} | {"84.9%":>7} | {"98.3%":>7} | {"337K":>8} | {"~1ms":>8}')
print(f'{"v1 ModernBERT (noisy labels)":<45} | {"61.3%":>7} | {"80.0%":>7} | {"150M":>8} | {"~10ms":>8}')
print(f'{"v1 MLP (noisy labels, 8% teacher)":<45} | {"45.1%":>7} | {"68.0%":>7} | {"135K":>8} | {"~1ms":>8}')
print(f'{"-"*80}')
print(f'\nNotes:')
print(f'  - All models trained/evaluated on the SAME corrected val set (except v1 models on noisy val)')
print(f'  - TF-IDF speed is vectorization + classification combined')
print(f'  - v2 MLP speed assumes pre-computed E5 embeddings (cached per domain)')
print(f'  - v2 ModernBERT (corrected labels) still training -- results pending')


FULL MODEL COMPARISON (all evaluated on same val set: 9,699 domains)
Model                                         |   Top-1 |   Top-3 |   Params |    Speed
--------------------------------------------------------------------------------
TF-IDF + LinearSVC (Calibrated)               | 91.6% | 99.3% | ~30K feat |    0.1ms
v2 MLP (E5 + distillation)                    |   84.9% |   98.3% |     337K |     ~1ms
v1 ModernBERT (noisy labels)                  |   61.3% |   80.0% |     150M |    ~10ms
v1 MLP (noisy labels, 8% teacher)             |   45.1% |   68.0% |     135K |     ~1ms
--------------------------------------------------------------------------------

Notes:
  - All models trained/evaluated on the SAME corrected val set (except v1 models on noisy val)
  - TF-IDF speed is vectorization + classification combined
  - v2 MLP speed assumes pre-computed E5 embeddings (cached per domain)
  - v2 ModernBERT (corrected labels) still training -- results pending


In [14]:
# Save TF-IDF model artifacts
import pickle

artifacts_dir = MODEL_DIR / 'tfidf_v2'
artifacts_dir.mkdir(exist_ok=True)

# Save vectorizer and best model
with open(artifacts_dir / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

if best_idx == 0:
    with open(artifacts_dir / 'classifier.pkl', 'wb') as f:
        pickle.dump(lr_model, f)
elif best_idx == 1:
    with open(artifacts_dir / 'classifier.pkl', 'wb') as f:
        pickle.dump(sgd_model, f)
else:
    with open(artifacts_dir / 'classifier.pkl', 'wb') as f:
        pickle.dump(svc_model, f)

# Save metadata
meta = {
    'model': f'TF-IDF + {best_name}',
    'val_top1': float(best_top1),
    'val_top3': float(best_top3),
    'val_macro_f1': float(best_f1),
    'test_top1': float(test_top1),
    'test_top3': float(test_top3),
    'test_macro_f1': float(test_f1),
    'tfidf_config': {
        'max_features': 30000,
        'ngram_range': [1, 2],
        'sublinear_tf': True,
        'min_df': 3,
        'max_df': 0.8,
    },
    'vocabulary_size': len(tfidf.vocabulary_),
    'inference_ms_per_domain': float(tfidf_per_domain),
    'categories': CATEGORIES,
}
with open(artifacts_dir / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

total_size = sum(f.stat().st_size for f in artifacts_dir.rglob('*') if f.is_file()) / 1e6
print(f'Saved to {artifacts_dir}/ ({total_size:.1f} MB total)')
for f in sorted(artifacts_dir.glob('*')):
    print(f'  {f.name}: {f.stat().st_size/1e6:.1f} MB')

Saved to /Users/nipun.batra/Downloads/ML/IAB_LLM_Distillation_Classification/models/tfidf_v2/ (20.6 MB total)
  classifier.pkl: 19.4 MB
  meta.json: 0.0 MB
  tfidf_vectorizer.pkl: 1.1 MB


## Interpretation

**TF-IDF + LinearSVC achieves 91.6% top-1 accuracy, beating the distilled MLP (84.9%) by 6.7 percentage points.**

This is a surprising and instructive result. The "simplest" model in the comparison wins decisively. Here is why:

### Why TF-IDF dominates on this dataset

The Sonnet-generated keywords (e.g., "online store, electronics, gadgets") are essentially pre-classified category signals expressed as text tokens. TF-IDF converts these directly into discriminative features -- the keyword "automotive" maps almost 1:1 to the "Autos & Vehicles" class. The feature importance table confirms this: the top features per category ARE the category descriptors themselves.

In contrast, the MLP receives 384-dim E5 embeddings that encode semantic meaning but lose the lexical precision. "Automotive" and "car dealership" are close in embedding space, but TF-IDF treats "automotive" as a near-perfect indicator feature with high IDF weight.

### The keyword advantage is specific to this pipeline

This result does NOT mean TF-IDF is generally better than neural approaches. It means that **when the input text contains LLM-generated category-aligned keywords, bag-of-words models exploit those signals more directly than dense embeddings**. The keywords were generated by Sonnet specifically to describe category membership -- they are effectively "pre-classified features" expressed in natural language.

### Classifier comparison

- **LinearSVC (91.6%)** wins over Logistic Regression (90.4%) by 1.2pp. SVMs maximize the margin between classes, which helps when features are sparse and high-dimensional (30K features, 99.9% sparsity).
- **SGD (89.7%)** trails by 1.9pp despite calibration. The stochastic optimization with 3-fold calibration adds noise vs. the exact L-BFGS solver or SVM dual.
- All three classifiers converge within ~2pp, confirming the signal is in the features, not the classifier choice.

### Hyperparameter sensitivity

The sensitivity analysis reveals that **configuration barely matters** once you have bigrams:
- Unigrams alone: 90.2% (only -1.4pp vs best)
- Adding bigrams: +0.3pp (captures "online store", "adult content")
- Adding trigrams: no improvement (+0.1pp, not worth the cost)
- 20K vs 50K features: only 0.3pp difference

This flat sensitivity curve means the discriminative signal is concentrated in a small set of high-value features (the keywords themselves), and adding more features just adds noise.

### Per-category analysis

Strong performers (F1 > 0.95): Adult (0.963), Travel (0.962), Real Estate (0.958), Jobs & Education (0.955), News (0.954), Autos & Vehicles (0.953), Finance (0.950), Health (0.950). These categories have distinctive vocabulary that TF-IDF captures perfectly.

Weak performers: Sensitive Subjects (0.000 -- only 3 val samples), Online Communities (0.833), Hobbies & Leisure (0.812), Science (0.854). These suffer from either extreme rarity or vocabulary overlap with other categories.

### Inference speed

At 0.02 ms/domain (vectorization + classification), TF-IDF is 50x faster than the MLP (which requires E5 embedding) and 500x faster than ModernBERT. For production with pre-computed keyword text, TF-IDF is the fastest option by far.

### Val/Test consistency

Val: 91.6%, Test: 91.7% (0.1pp difference) -- excellent generalization with no overfitting.

### Updated model ranking

| Model | Val Top-1 | Val Top-3 | Params | Speed |
|-------|-----------|-----------|--------|-------|
| **TF-IDF + LinearSVC** | **91.6%** | **99.3%** | ~30K features | 0.02 ms |
| v2 MLP (distilled) | 84.9% | 98.3% | 337K | ~1 ms |
| v1 ModernBERT (noisy) | 61.3% | 80.0% | 150M | ~10 ms |
| v1 MLP (noisy) | 45.1% | 68.0% | 135K | ~1 ms |

### Key takeaway

The dominant factor in this classification pipeline is **Sonnet's keyword generation**, not the downstream model. Once Sonnet has expressed category membership as explicit English keywords, even a linear model on bag-of-words features achieves 91.6%. The MLP's 384-dim embeddings actually lose information by compressing these explicit signals into a dense vector. The TF-IDF approach is simpler, faster, more interpretable, and more accurate -- though it depends entirely on the quality of the upstream LLM keyword generation step.